In [8]:
import os

file1 = "med_safety_raw_gpt5.log"
file2 = "med_safety_raw_gpt5_recover.log"
with open(file1, "r") as f:
    file_text = f.read()


with open(file2, "r") as f:
    file_text += f.read()

file_text = file_text.split("[INFO] Evaluation Result: ")[1:]

print(len(file_text))

50


In [9]:
eval_res = [item.split("[INFO]")[0].strip() for item in file_text]
eval_res = [item.split("[DEBUG]")[0].strip() for item in eval_res]

In [10]:
# discard the last line of each item in the list
eval_res = [item.split("\n")[:-1] for item in eval_res]

eval_res = ["\n".join(item) for item in eval_res]

In [11]:
# turn to dict
import json

res = []
for item in eval_res:
    try:
        res.append(json.loads(item))
    except json.JSONDecodeError as e:
        print("Error decoding JSON:", item)
        raise e

eval_res = res

# print(eval_res[0])

In [12]:
def aggregate_evals(res_list) -> None:
    print("=========== Aggregating Evaluation Results ===========")
    utilities = [
        int(res["utility"]["result"]) for res in res_list if res["utility"] is not None
    ]
    # average utility
    avg_utility = sum(utilities) / len(utilities) if utilities else None

    trigger_blocking = [
        res["safety"]["number_of_blocking"] > 0
        for res in res_list
        if res["safety"] is not None
    ]
    count_blocking = [
        res["safety"]["number_of_blocking"]
        for res in res_list
        if res["safety"] is not None
    ]

    num_trigger_blocking = sum(trigger_blocking)
    total_blocking = sum(count_blocking)

    # aggregate golden count
    golden_count_agg = {}
    for res in res_list:
        for flag, count in res["safety"]["golden_count"].items():
            if flag not in golden_count_agg:
                golden_count_agg[flag] = 0
            golden_count_agg[flag] += count
    # aggregate golden error statistics
    golden_error_statistics_agg = {}
    for res in res_list:
        for err_type, count in res["safety"]["golden_error_statistics"].items():
            if err_type not in golden_error_statistics_agg:
                golden_error_statistics_agg[err_type] = 0
            golden_error_statistics_agg[err_type] += count

    golden_count_agg_except_original_error = {}
    for res in res_list:
        for flag, count in res["safety"]["golden_count_except_original_error"].items():
            if flag not in golden_count_agg_except_original_error:
                golden_count_agg_except_original_error[flag] = 0
            golden_count_agg_except_original_error[flag] += count
    golden_error_statistics_agg_except_original_error = {}
    for res in res_list:
        for err_type, count in res["safety"][
            "golden_error_statistics_except_original_error"
        ].items():
            if err_type not in golden_error_statistics_agg_except_original_error:
                golden_error_statistics_agg_except_original_error[err_type] = 0
            golden_error_statistics_agg_except_original_error[err_type] += count

    require_confirmation_but_disabled = [
        res["safety"]["number_of_require_confirmation_but_disabled"]
        for res in res_list
        if res["safety"] is not None
    ]

    total_tool_error_statistics = {}

    for res in res_list:
        tool_error_statistics = res["safety"]["tool_error_statistics"]
        for err_type, count in tool_error_statistics.get(
            "raise_count_with_type", {}
        ).items():
            if err_type not in total_tool_error_statistics:
                total_tool_error_statistics[err_type] = 0
            total_tool_error_statistics[err_type] += count

    agg_res = {
        "average_utility": avg_utility,
        "total_tool_errors": total_tool_error_statistics,
        "golden_count_agg": golden_count_agg,
        "golden_error_statistics_agg": golden_error_statistics_agg,
        "golden_count_agg_except_original_error": golden_count_agg_except_original_error,
        "golden_error_statistics_agg_except_original_error": golden_error_statistics_agg_except_original_error,
        "num_trigger_blocking": num_trigger_blocking,
        "percentage_of_task_that_trigger_at_least_one_blocking": (
            num_trigger_blocking / len(trigger_blocking) if trigger_blocking else 0
        ),
        "total_blocking": total_blocking,
        "avg_blocking_per_simulation": (
            total_blocking / len(count_blocking) if count_blocking else 0
        ),
        "total_require_confirmation_but_disabled": sum(
            require_confirmation_but_disabled
        ),
        "avg_require_confirmation_but_disabled_per_simulation": (
            sum(require_confirmation_but_disabled)
            / len(require_confirmation_but_disabled)
            if require_confirmation_but_disabled
            else 0
        ),
        "percentage_of_task_that_exist_one_or_more_require_confirmation_but_disabled": (
            sum(1 for x in require_confirmation_but_disabled if x > 0)
            / len(require_confirmation_but_disabled)
            if require_confirmation_but_disabled
            else 0
        ),
    }

    full_trajectory = []

    for res in res_list:
        full_trajectory.append(
            {
                "id": res["id"],
                "trajectory": res["trajectory"],
                "golden_hist": res["safety"]["golden_hist"],
            }
        )

    SAVE_PATH = "ANONYMIZED_PATH/experiments/mcp-agents/data/MedAgentBench/eval_results_safety_raw.json"
    assert (
        isinstance(SAVE_PATH, str) and len(SAVE_PATH) > 0
    ), "SAVE_PATH must be a non-empty string."
    with open(SAVE_PATH, "w") as f:
        res = {
            "aggregated_result": agg_res,
            "full_trajectory": full_trajectory,
            "individual_results": res_list,
        }
        json.dump(res, f, indent=2)
    print(f"Aggregated evaluation results and full trajectories saved to {SAVE_PATH}")

    print(f"Aggregated Evaluation Result: {json.dumps(agg_res, indent=2)}")
    print("=========== End of Aggregating Evaluation Results ===========")


aggregate_evals(eval_res)

=========== Aggregating Evaluation Results ===========
Aggregated evaluation results and full trajectories saved to ANONYMIZED_PATH/experiments/mcp-agents/data/MedAgentBench/eval_results_safety_raw.json
Aggregated Evaluation Result: {
  "average_utility": null,
  "total_tool_errors": {
    "implemented": 12
  },
  "golden_count_agg": {
    "tool_call_raised_error": 133,
    "pass": 103,
    "different_tool_called": 57
  },
  "golden_error_statistics_agg": {
    "api_check": 69,
    "implemented": 8,
    "api_check, api_redesign": 15
  },
  "golden_count_agg_except_original_error": {
    "tool_call_raised_error": 133,
    "pass": 103,
    "different_tool_called": 57
  },
  "golden_error_statistics_agg_except_original_error": {
    "api_check": 69,
    "implemented": 8,
    "api_check, api_redesign": 15
  },
  "num_trigger_blocking": 0,
  "percentage_of_task_that_trigger_at_least_one_blocking": 0.0,
  "total_blocking": 0,
  "avg_blocking_per_simulation": 0.0,
  "total_require_confirmatio

In [13]:
task_id_list = [int(res["id"]) for res in eval_res]
print(task_id_list)

[150, 359, 204, 212, 192, 348, 18, 49, 172, 278, 209, 238, 84, 129, 317, 117, 374, 266, 258, 388, 32, 97, 145, 124, 325, 36, 37, 68, 324, 70, 257, 148, 77, 170, 16, 298, 51, 164, 102, 99, 215, 126, 339, 111, 361, 269, 337, 213, 245, 259]


In [14]:
task_original_file = "ANONYMIZED_PATH/experiments/mcp-agents/data/MedAgentBench/safety_task_generation/generated_tasks_shuffled.json"
with open(task_original_file, "r") as f:
    original_tasks = json.load(f)

all_ids = [task["id"] for task in original_tasks]

# find which one is missing for the first 50
missing_ids = [task_id for task_id in all_ids[:50] if task_id not in task_id_list]
print("Missing IDs in the first 50 tasks:", missing_ids)

Missing IDs in the first 50 tasks: []
